# 🚀 Using the Mini LLM

The companion notebook **`v1_builder.ipynb`** built and trained a 0.9 M-parameter, GPT-style model on the Python standard library, and saved it to **`mini_llm_v1.pt`**. This notebook **loads that checkpoint and puts it to work**, with no retraining.

### 🗺️ What this notebook does

| # | Step | Why |
|---|---|---|
| 1 | Load the checkpoint | See what was saved |
| 2 | Rebuild the tokenizer | Text ↔ token IDs, from the saved merges |
| 3 | Rebuild the model | Recreate the architecture and load the trained weights |
| 4 | Check the reload | Held-out loss should match the end of training (≈ 2.34) |
| 5 | Look at next-token predictions | What the model thinks comes next |
| 6 | Generate code | Complete prompts one token at a time |
| 7 | Change the temperature | How sampling settings change the output |
| 8 | Try your own prompt | A ready-made helper |


> **A checkpoint stores numbers, not code.** `mini_llm_v1.pt` holds the weights, the config and the tokenizer tables, but **not** the model's Python classes or the tokenizer's pre-split regex. Both are redefined below **exactly** as in the builder. If either drifts, the weights won't load or the token IDs won't line up.

**Requirements:** run from the `v1/` folder (next to `mini_llm_v1.pt` and `corpus.txt`), with Python 3.14 + PyTorch.

---
## 1️⃣ Load the checkpoint

The checkpoint is a plain Python dict written by `torch.save` at the end of training:

| Key | Contents |
|---|---|
| `model` | the `state_dict`: every trained weight table, by name |
| `cfg` | hyperparameters (`vocab`, `width`, `layers`, `heads`, `inner`, `seq`, `batch`) |
| `merges` | the 255 BPE merges `(a, b) → new ID`, **in the order they were learned** |
| `vocab` | token ID → the bytes it stands for |

In [ ]:
import math, re, time
import torch
import torch.nn as nn
import torch.nn.functional as F

CKPT = "mini_llm_v1.pt"
device = "mps" if torch.backends.mps.is_available() else "cpu"    # Apple GPU if present

# map_location moves every saved tensor straight onto our device
ckpt = torch.load(CKPT, map_location=device)
cfg, merges, vocab = ckpt["cfg"], ckpt["merges"], ckpt["vocab"]

print("torch:", torch.__version__, "| device:", device)
print("checkpoint keys:", list(ckpt))
print("cfg:", cfg)
print("merges:", len(merges), "| vocab:", len(vocab), "| weight tensors:", len(ckpt["model"]))

**Output**

```code
torch: 2.15.0 | device: mps
checkpoint keys: ['model', 'cfg', 'merges', 'vocab']
cfg: {'vocab': 512, 'width': 128, 'layers': 4, 'heads': 4, 'inner': 512, 'seq': 256, 'batch': 16}
merges: 255 | vocab: 512 | weight tensors: 35
```

> 💡 **What to notice:** The saved `cfg` has `seq=256, batch=16`, the values training actually used, not the 128/32 first set in section 1 of the builder.

---
## 2️⃣ Rebuild the tokenizer

The saved `merges` and `vocab` are the tokenizer's *data*. The *logic* is rebuilt here:

- **`SPLIT`**: the pre-split regex. It is **not** in the checkpoint and must match the builder exactly.
- **`encode`**: split into chunks, then replay the merges in the order they were learned (earliest first).
- **`decode`**: join the bytes of each ID and decode as UTF-8.

The cell checks the result two ways: encoding the same example line as the builder must give the **same IDs**, and decoding must return the original text.

In [ ]:
# ── NOT stored in the checkpoint: must match v1_builder.ipynb exactly ────────
SPLIT = re.compile(r" ?[A-Za-z_]+| ?\d+| ?[^\sA-Za-z_\d]+|\s+(?!\S)|\s+")
EOF_MARK = "<|eof|>"
EOF_ID = len(vocab) - 1                          # the last ID is the end-of-file marker
assert vocab[EOF_ID] == EOF_MARK.encode()

rank = {pair: i for i, pair in enumerate(merges)}   # dicts keep insertion order = learning order
cache = {}                                          # chunk -> token IDs

def encode_chunk(c):
    """Turn one pre-split chunk into token IDs by replaying the learned merges."""
    if c in cache: return cache[c]
    ids = list(c.encode("utf-8"))                   # start from raw bytes
    while len(ids) > 1:
        pair = min(zip(ids, ids[1:]), key=lambda p: rank.get(p, 1e9))  # earliest-learned pair
        if pair not in merges: break                # nothing left to merge
        new, out, i = merges[pair], [], 0
        while i < len(ids):
            if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
                out.append(new); i += 2
            else:
                out.append(ids[i]); i += 1
        ids = out
    cache[c] = ids
    return ids

def encode(s): return [i for c in SPLIT.findall(s) for i in encode_chunk(c)]                  # text -> IDs
def decode(ids): return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")    # IDs -> text

# Same example as the builder: the IDs must be identical
sample = "    def __init__(self, name):"
ids = encode(sample)
print("IDs:   ", ids)
print("pieces:", [vocab[i].decode() for i in ids])
print("matches builder:", ids == [258, 338, 457, 261, 294, 309, 40, 277, 44, 440, 302])
print("round trip ok:  ", decode(ids) == sample)

**Output**

```code
IDs:    [258, 338, 457, 261, 294, 309, 40, 277, 44, 440, 302]
pieces: ['   ', ' def', ' __', 'in', 'it', '__', '(', 'self', ',', ' name', '):']
matches builder: True
round trip ok:   True
```

> 💡 **What to notice:** The IDs match the builder exactly, so the regex and the merge order were restored correctly.

---
## 3️⃣ Rebuild the model and load the weights

The `Block` and `MiniLLM` classes are copied from the builder unchanged. Their sizes come from the saved `cfg`, so the shapes line up with the checkpoint.

`load_state_dict` is **strict** by default: every weight name and shape must match exactly, otherwise it raises an error. A clean load proves the architecture was rebuilt correctly.

In [ ]:
V, W, L, I = cfg["vocab"], cfg["width"], cfg["layers"], cfg["inner"]
H = cfg["heads"]; D = W // H                      # 4 heads x 32 fields
SEQ = cfg["seq"]                                  # context window used in training (256)

def rope(t, T):
    """Rotary position embedding: rotate field pairs by a position-dependent angle."""
    half = D // 2
    freqs = 1.0 / (10000 ** (torch.arange(half, device=t.device) / half))
    ang = torch.arange(T, device=t.device)[:, None] * freqs[None, :]
    t1, t2 = t[..., :half], t[..., half:]
    return torch.cat([t1 * ang.cos() - t2 * ang.sin(), t1 * ang.sin() + t2 * ang.cos()], dim=-1)

class Block(nn.Module):                          # one layer: Mix + Refine
    def __init__(self):
        super().__init__()
        self.norm1, self.norm2 = nn.RMSNorm(W), nn.RMSNorm(W)
        self.Wq, self.Wk, self.Wv, self.Wo = [nn.Linear(W, W, bias=False) for _ in range(4)]   # attention tables
        self.Up, self.Down = nn.Linear(W, I, bias=False), nn.Linear(I, W, bias=False)          # feed-forward tables

    def forward(self, g):
        B, T, _ = g.shape
        h = self.norm1(g)
        sp = lambda t: t.view(B, T, H, D).transpose(1, 2)             # split into heads
        q, k, v = sp(self.Wq(h)), sp(self.Wk(h)), sp(self.Wv(h))
        q, k = rope(q, T), rope(k, T)
        m = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # causal attention
        g = g + self.Wo(m.transpose(1, 2).reshape(B, T, W))           # Mix + residual
        g = g + self.Down(F.gelu(self.Up(self.norm2(g))))             # Refine + residual
        return g

class MiniLLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, W)                                  # Embedding TABLE
        self.blocks = nn.ModuleList([Block() for _ in range(L)])       # L stacked layers
        self.norm = nn.RMSNorm(W)                                      # final norm
        self.out = nn.Linear(W, V, bias=False)                         # Output TABLE

    def forward(self, ids):
        g = self.emb(ids)
        for b in self.blocks:
            g = b(g)
        return self.out(self.norm(g))                                  # logits: (batch, seq, vocab)

model = MiniLLM().to(device)
result = model.load_state_dict(ckpt["model"])     # strict: every name + shape must match
model.eval()                                      # inference mode (no training behaviour)

print("load:", result)
print("tables:", len(list(model.parameters())),
      "| params:", f"{sum(p.numel() for p in model.parameters()):,}")

**Output**

```code
load: <All keys matched successfully>
tables: 35 | params: 918,656
```

> 💡 **What to notice:** `<All keys matched successfully>`: all 35 tables and 918,656 parameters, the same as in the builder.

---
## 4️⃣ Check the reload: held-out loss

A clean load proves the shapes match, but not that the model *behaves* as it did after training. To check that, the cell rebuilds the exact held-out split (the last 10% of tokens of `corpus.txt`) and measures the loss.

During training the loss was estimated on 20 *random* batches. Here it is computed over **every** non-overlapping 256-token window, so it is a precise, repeatable number. Two baselines put it in context:

- an **untrained** model with random weights (≈ ln 512 ≈ 6.24, pure guessing)
- the **trained** model, which should land close to the **2.34** reported at the end of training

The loss is also converted into easier-to-read units:
- **perplexity** = $e^{\text{loss}}$: the model is "as unsure as choosing among *N* equally likely tokens"
- **bits per character** = $\text{loss} / \ln 2 \;/\; \text{chars per token}$

In [ ]:
# Rebuild the exact token stream and train / held-out split used in the builder
corpus = open("corpus.txt", encoding="utf-8").read()
t0 = time.time()
all_ids = []
for doc in corpus.split(f"\n{EOF_MARK}\n"):
    all_ids += encode(doc) + [EOF_ID]
data = torch.tensor(all_ids, dtype=torch.long)
val_ids = data[int(0.9 * len(data)):]             # last 10% = held-out
chars_per_token = len(corpus) / len(all_ids)
print(f"tokens: {len(all_ids):,} | held-out: {len(val_ids):,} | encoded in {time.time()-t0:.0f}s")

@torch.no_grad()
def heldout_loss(net, ids, batch=32):
    """Average next-token loss over every non-overlapping SEQ-token window."""
    n = (len(ids) - 1) // SEQ                     # number of full windows
    x = ids[: n * SEQ].view(n, SEQ)               # inputs
    y = ids[1 : n * SEQ + 1].view(n, SEQ)         # targets = inputs shifted by 1
    total = 0.0
    for i in range(0, n, batch):
        xb, yb = x[i:i+batch].to(device), y[i:i+batch].to(device)
        total += F.cross_entropy(net(xb).view(-1, V), yb.view(-1), reduction="sum").item()
    return total / (n * SEQ), n

torch.manual_seed(0)
untrained = MiniLLM().to(device).eval()           # same architecture, random weights

print(f"\n{'model':10} {'loss':>6} {'perplexity':>11} {'bits/char':>10}")
for name, net in [("untrained", untrained), ("trained", model)]:
    loss, n = heldout_loss(net, val_ids)
    print(f"{name:10} {loss:6.3f} {math.exp(loss):11.1f} {loss / math.log(2) / chars_per_token:10.2f}")
print(f"\n({n:,} windows x {SEQ} tokens; uniform guess = ln({V}) = {math.log(V):.3f})")

**Output**

```code
tokens: 3,906,008 | held-out: 390,601 | encoded in 1s

model        loss  perplexity  bits/char
untrained   6.426       617.4       4.30
trained     2.362        10.6       1.58

(1,525 windows x 256 tokens; uniform guess = ln(512) = 6.238)
```

> 💡 **What to notice:** The reloaded model scores **2.362**, close to the **2.337** from training (which was estimated on 20 random batches), so the reload is correct. A perplexity of **10.6** means that at each step the model is about as unsure as choosing among ~11 tokens, compared with ~617 before training. The untrained loss (6.43) is a little above ln 512 because random weights are *confidently* wrong rather than perfectly uniform.

---
## 5️⃣ What does the model think comes next?

Before sampling whole programs, it helps to look at a **single prediction**: the probability the model gives each possible next token. A good model puts most of its probability on a few sensible choices.

### ⚠️ Never end a prompt in whitespace
`SPLIT` glues whitespace to what **follows** it: a space joins the next word (`" os"`, `" self"`), and a newline joins the next line's indentation (`"\n   "`). A prompt that *ends* in whitespace therefore leaves a lone token that meant something different in training:

| Prompt ends with | Last token | What the model saw in training |
|---|---|---|
| `"import"` | `'import'` | a word, so the next token continues normally ✅ |
| `"import "` | `' '` | a bare space before unusual tokens, so it guesses capital letters ❌ |
| `"def f(x):"` | `'):'` | the model picks the newline **and** the indentation itself ✅ |
| `"def f(x):\n"` | `'\n'` | a bare newline only ever came before **column-0** code, so the body is never indented ❌ |

The last four prompts below show this.

In [ ]:
@torch.no_grad()
def show_next(prompt, k=6):
    """Print the k most likely next tokens after `prompt`, with a small bar chart."""
    ids = torch.tensor([encode(prompt)], device=device)
    probs = F.softmax(model(ids[:, -SEQ:])[0, -1], dim=-1)   # distribution after the last token
    top = probs.topk(k)
    print(f"{prompt!r}")
    for p, i in zip(top.values.tolist(), top.indices.tolist()):
        print(f"    {vocab[i].decode('utf-8', 'replace')!r:14} {p:6.1%}  {'█' * round(p * 40)}")

for prompt in ["for i in range", "    def __init__(self",
               "import", "import ",                                     # word vs trailing space
               "def read_config(path):", "def read_config(path):\n"]:   # colon vs trailing newline
    show_next(prompt)

**Output**

```code
'for i in range'
    's'             23.0%  █████████
    '('             13.2%  █████
    '\n   '         11.8%  █████
    '.'              8.8%  ████
    ' as'            3.7%  █
    ':'              3.4%  █
'    def __init__(self'
    ','             72.4%  █████████████████████████████
    '):'            23.4%  █████████
    ')'              2.9%  █
    '.'              0.8%  
    '_'              0.1%  
    '))'             0.1%  
'import'
    'er'            25.2%  ██████████
    'Error'          8.4%  ███
    'li'             6.4%  ███
    '_'              5.5%  ██
    's'              4.5%  ██
    ' '              2.9%  █
'import '
    'P'              8.9%  ████
    'E'              5.4%  ██
    'get'            4.8%  ██
    'C'              4.7%  ██
    'O'              4.5%  ██
    'li'             4.0%  ██
'def read_config(path):'
    '\n       '     30.0%  ████████████
    '\n   '         29.6%  ████████████
    '\n           '  22.2%  █████████
    '\n               '   6.8%  ███
    '\n                   '   2.6%  █
    '\n                    '   1.2%  
'def read_config(path):\n'
    '\n'            34.0%  ██████████████
    '#'             22.1%  █████████
    'def'            5.2%  ██
    'f'              3.9%  ██
    '<|eof|>'        3.7%  █
    '\n           '   2.4%  █
```

> 💡 **What to notice:** After `__init__(self` the model splits between `,` (72%) and `):` (23%), the two ways a method signature continues. **`import` vs `import `**: the trailing space scatters the probability over capital letters. **`:` vs `:\n`**: ending at the colon puts over 90% of the probability on newline + indentation tokens, while the bare `\n` expects column-0 code (`#`, `def`, even `<|eof|>`).

---
## 6️⃣ Generate code

Generation repeats one step: **predict → sample → append**, until `max_new` tokens or the model emits `<|eof|>`.

- **Context window**: only the last `SEQ = 256` tokens are fed back in, since that is all the model was trained to see.
- **Temperature**: divides the logits. Values below 1 sharpen the distribution, values above 1 flatten it, and `0` means **greedy** (always take the top token).
- **Top-k**: removes every token outside the *k* most likely before sampling.
- **`seed`**: makes a sample repeatable.

`complete()` prints the prompt and then the model's continuation, with a line between them.

In [ ]:
@torch.no_grad()
def generate(prompt, max_new=150, temperature=0.8, top_k=40, seed=None):
    """Continue `prompt` one token at a time and return prompt + continuation."""
    if seed is not None: torch.manual_seed(seed)
    ids = torch.tensor([encode(prompt)], device=device)
    for _ in range(max_new):
        logits = model(ids[:, -SEQ:])[:, -1, :]                  # scores for the next token
        if temperature == 0:
            nxt = logits.argmax(dim=-1, keepdim=True)            # greedy: the single best token
        else:
            logits = logits / temperature
            if top_k:
                kth = torch.topk(logits, top_k).values[:, [-1]]
                logits[logits < kth] = -float("inf")             # keep only the top-k
            nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        if nxt.item() == EOF_ID: break                           # model says "file over"
        ids = torch.cat([ids, nxt], dim=1)
    return decode(ids[0].tolist())

def complete(prompt, **kw):
    """Print the prompt, a divider, then what the model wrote."""
    out = generate(prompt, **kw)
    print(prompt.rstrip("\n"))
    print("─" * 22 + " model wrote " + "─" * 22)
    print(out[len(prompt):].strip("\n"))
    print("═" * 57)

prompts = [                                       # each ends at ":" (no trailing whitespace)
    "class Stack:\n    def __init__(self):",
    "def read_config(path):",
    "    def __repr__(self):",
]
for i, p in enumerate(prompts):
    complete(p, max_new=100, seed=i)

**Output**

```code
class Stack:
    def __init__(self):
────────────────────── model wrote ──────────────────────
        return floatstring(self)

    def time(self, tuplees):
        self._shutdown = timeout
        self.__doc__ = localcaltime

    def __getitem__(self):
        """
        Return the speed context mana."""
        self._sslocal = True
        self.__doc
═════════════════════════════════════════════════════════
def read_config(path):
────────────────────── model wrote ──────────────────────
        """Repath = repath.join()
        exp = ''.join(path)
        # list.
        for i in list of paths:
            if dirname.path is obout:
                register('path', repr(path[0])
                listener.copy()
                path = m.group
═════════════════════════════════════════════════════════
    def __repr__(self):
────────────────────── model wrote ──────────────────────
                _warnings.repr(self._was_selections.clear(), write_stack, self._shapes)
                self._state
        self._flush()

        def __repr__(self):
            self._callmended_own_state()

        # Console: Python st
═════════════════════════════════════════════════════════
```

> 💡 **What to notice:** Indentation, `self.` attributes, docstrings and nested blocks look right, but the content is nonsense and has syntax errors (`if dirname.path is obout:`). The model usually indents **8 spaces** (method-body depth), because methods inside classes are the most common kind of function in the standard library.

---
## 7️⃣ Change the temperature

The same prompt and seed, with only the temperature changed:

- **0 (greedy)**: deterministic. Small models often fall into **repetition loops** this way.
- **0.5**: conservative. The output stays close to the most common patterns.
- **0.8**: the default. More variety, still mostly well-formed.
- **1.2**: adventurous. More made-up names and broken syntax.

In [ ]:
prompt = "def parse_args(argv):"
for temp in [0, 0.5, 0.8, 1.2]:
    print(f"▶ temperature = {temp}")
    print(generate(prompt, max_new=70, temperature=temp, seed=0))
    print()

**Output**

```code
▶ temperature = 0
def parse_args(argv):
        """Return the list of the list of the list of the list of the list.

        Optional arguments are are aren't the list of the list.
        """
        if not isinstance(re

▶ temperature = 0.5
def parse_args(argv):
        """Close the CREST, and version of the module."""
        url = self._version
        if self._sslobj.current_indentations:
            return self._map._PROSIX

▶ temperature = 0.8
def parse_args(argv):
        _float = get_state[param]
        if max(attr < 4):
            return self._to_minute
            else:
                self.traverse(new_state, self._sslobj)

            self.

▶ temperature = 1.2
def parse_args(argv):
        _float1 = _os.splparserver = TOMBCHAL {1}
      stack_ass amlist (heading) does the NOTE:
        FailTable replace filters
```

> 💡 **What to notice:** Greedy decoding gets stuck in a loop (*"the list of the list of the list…"*). 0.5 and 0.8 stay readable, and at 1.2 the syntax falls apart (`TOMBCHAL {1}`, mismatched indentation).

---
## 8️⃣ Try your own prompt

Edit the prompt and the settings below, and run the cell again.

**Tips**
- Start with a line that sets up structure, such as a `def`/`class` header.
- **End the prompt at the `:`**, not after a space or newline (see section 5). The model adds the newline and indentation itself.
- The model has seen only the standard library, so stdlib-style code (`os`, `re`, `self._private`, `raise ValueError(...)`) works best.
- Remove `seed=` to get a different sample each run.

In [ ]:
complete(
    "import re\n\n\ndef tokenize(text):",     # ← your prompt here (end at the ":")
    max_new=120,
    temperature=0.7,
    top_k=40,
    seed=7,
)

**Output**

```code
import re


def tokenize(text):
────────────────────── model wrote ──────────────────────
    """Invalid for impath in this list of types."""
    for exp in range(path, path, listsize=_Path):
        flags = list(path)
        if len(path) > while list:
            list(path)

        list = Path(path_http_hi, path_history=path,
                                
═════════════════════════════════════════════════════════
```

> 💡 **What to notice:** With `import re` above it, the `def` is clearly at module level, and this time the model correctly picks a **4-space** indent.

---
## ✅ Summary

- The checkpoint restores a **working model**: weights, config and tokenizer, with no retraining.
- The **model code and the pre-split regex are not saved** and must be kept in sync with the builder. In a real project they would live in a shared `.py` module that both notebooks import.
- The reloaded model's held-out loss matches training, so the round trip (train → save → load → use) is verified.

### ⚠️ Limitations of this model
- **It copies style, not meaning**: the output looks like Python but would not run.
- **Short memory**: a 256-token window is about 550 characters, so it cannot keep track of a whole file.
- **Narrow knowledge**: it has seen only the Python standard library.

### 🚀 Ideas
- Move `encode`/`decode`/`MiniLLM` into a shared `mini_llm.py` so neither notebook redefines them.
- Add a **KV-cache** so each new token doesn't reprocess the whole context.
- Measure how many generated samples pass `ast.parse`.